# 01 — Data Ingestion

Downloads and prepares all data for the regime model:

**Sources:**
- **CRSP** (via WRDS): S&P 500 daily index prices
- **FRED API**: Yield curve (T10Y3M), WTI oil, credit spread (BAA10Y), VIX, 3-month yield (TB3MS)
- **Ken French Data Library**: FF5 factors + Momentum, 12 Industry Portfolios

**Output:** `data` DataFrame with 8 macro z-score variables (468 months, 1986-2024)

Based on: *Regimes, Systematic Models and the Power of Prediction* — Man Group (March 2025)

In [2]:
import wrds
conn = wrds.Connection()

sp500 = conn.raw_sql("""
    SELECT caldt, spindx
    FROM crsp.dsp500
    WHERE caldt >= '1926-01-01'
    ORDER BY caldt
""")

sp500.to_csv("sp500_crsp.csv", index=False)
print(sp500.shape)
print(sp500.head())

Enter your WRDS username [daniel]:gusigokida
Enter your password:········


OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: SSL connection has been closed unexpectedly

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [9]:
"""
Regime Model - Data Ingestion
===============================
Variables:
    1. Market         - S&P 500 (CRSP csv)
    2. Yield Curve    - US 10yr minus 3-month yield (FRED)
    3. Oil            - WTI Crude (FRED)
    4. Gold           - FRED IQ12260 (testing)
    5. Credit Spread  - BAA minus 10yr (FRED BAA10Y)
    6. Monetary Policy- US 3-month yield (FRED)
    7. Volatility     - VIX (FRED) stitched with realized vol
    8. Stock-Bond Corr- Rolling 3yr correlation S&P500 vs 10yr yield (computed)
"""
import os
import pandas as pd
import numpy as np
from fredapi import Fred

# ── Config ────────────────────────────────────────────────────────────────────
FRED_API_KEY = "db0e8230b3719503f33a50ae27dc9f2a"
START_DATE   = "1926-01-01"
END_DATE     = None

fred = Fred(api_key=FRED_API_KEY)


# ── Helper ────────────────────────────────────────────────────────────────────
def fetch_fred(series_id: str, name: str) -> pd.Series:
    s = fred.get_series(series_id, observation_start=START_DATE)
    s.name = name
    return s.resample("M").last().ffill()


# ── 1. S&P 500 from CRSP ─────────────────────────────────────────────────────
print("Fetching S&P 500...")
sp500_raw           = pd.read_csv("data/sp500_crsp.csv", parse_dates=["caldt"], index_col="caldt")
sp500_daily         = sp500_raw["spindx"].dropna()
sp500_returns_daily = sp500_daily.pct_change().dropna()
sp500_monthly       = sp500_daily.resample("M").last().rename("sp500")


# ── 2. Yield Curve ────────────────────────────────────────────────────────────
print("Fetching yield curve...")
gs10        = fetch_fred("GS10",  "gs10")
tb3ms       = fetch_fred("TB3MS", "tb3ms")
yield_curve = (gs10 - tb3ms).rename("yield_curve")


# ── 3. Oil ────────────────────────────────────────────────────────────────────
print("Fetching WTI oil...")
oil = fetch_fred("DCOILWTICO", "oil")


# ── 4. Gold & Credit Spread (replacing copper) ───────────────────────────────
print("Fetching gold and credit spread...")
try:
    gold = fetch_fred("IQ12260", "gold")
    print("gold:         ", gold.index[0], gold.shape)
except Exception as e:
    print(f"gold FAILED: {e}")
    gold = None

credit_spread = fetch_fred("BAA10Y", "credit_spread")
print("credit_spread:", credit_spread.index[0], credit_spread.shape)


# ── 5. Monetary Policy ───────────────────────────────────────────────────────
monetary_policy = tb3ms.rename("monetary_policy")


# ── 6. Volatility ────────────────────────────────────────────────────────────
print("Fetching VIX...")
vix          = fetch_fred("VIXCLS", "vix")
realized_vol = (
    sp500_returns_daily
    .rolling(21)
    .std()
    .multiply(np.sqrt(252) * 100)
    .resample("M")
    .last()
    .rename("realized_vol")
)
volatility = vix.combine_first(realized_vol).rename("volatility")


# ── 7. Stock-Bond Correlation ─────────────────────────────────────────────────
print("Computing stock-bond correlation...")
gs10_daily         = fred.get_series("GS10", observation_start=START_DATE)
gs10_returns_daily = gs10_daily.resample("D").last().ffill().pct_change().dropna()

corr_df = pd.DataFrame({
    "sp500": sp500_returns_daily,
    "gs10":  gs10_returns_daily
}).dropna()

stock_bond_corr = (
    corr_df["sp500"]
    .rolling(504)                           # 2yr window (~504 trading days)
    .corr(corr_df["gs10"])
    .resample("M")
    .last()
    .rename("stock_bond_corr")
)


# ── Diagnostics ───────────────────────────────────────────────────────────────
print("\n--- Series diagnostics ---")
print("sp500_monthly:  ", sp500_monthly.shape,   sp500_monthly.index[0])
print("yield_curve:    ", yield_curve.shape,     yield_curve.index[0])
print("oil:            ", oil.shape,             oil.index[0])
if gold is not None:
    print("gold:           ", gold.shape,         gold.index[0])
print("credit_spread:  ", credit_spread.shape,   credit_spread.index[0])
print("monetary_policy:", monetary_policy.shape)
print("volatility:     ", volatility.shape)
print("stock_bond_corr:", stock_bond_corr.shape)


# ── Combine ───────────────────────────────────────────────────────────────────
print("\nCombining all variables...")

series_to_concat = [
    sp500_monthly,
    yield_curve,
    oil,
    credit_spread,
    monetary_policy,
    volatility,
    stock_bond_corr,
]

# only include gold if it loaded successfully
if gold is not None:
    series_to_concat.insert(3, gold)

data = pd.concat(series_to_concat, axis=1)
data.index = pd.to_datetime(data.index)
data = data.loc[START_DATE:].dropna()

print(f"\nFinal dataset shape: {data.shape}")
print(f"Date range: {data.index[0].date()} → {data.index[-1].date()}")
print(f"Columns: {list(data.columns)}")
print(f"\nMissing values:\n{data.isnull().sum()}")
print(f"\nFirst rows:\n{data.head()}")

os.makedirs("data", exist_ok=True)
data.to_csv("data/regime_raw_data.csv")
print("\nSaved to data/regime_raw_data.csv")

Fetching S&P 500...
Fetching yield curve...
Fetching WTI oil...
Fetching gold and credit spread...
gold:          1984-12-31 00:00:00 (495,)
credit_spread: 1986-01-31 00:00:00 (484,)
Fetching VIX...
Computing stock-bond correlation...

--- Series diagnostics ---
sp500_monthly:   (750,) 1962-07-31 00:00:00
yield_curve:     (1107,) 1934-01-31 00:00:00
oil:             (484,) 1986-01-31 00:00:00
gold:            (495,) 1984-12-31 00:00:00
credit_spread:   (484,) 1986-01-31 00:00:00
monetary_policy: (1107,)
volatility:      (766,)
stock_bond_corr: (750,)

Combining all variables...

Final dataset shape: (468, 8)
Date range: 1986-01-31 → 2024-12-31
Columns: ['sp500', 'yield_curve', 'oil', 'gold', 'credit_spread', 'monetary_policy', 'volatility', 'stock_bond_corr']

Missing values:
sp500              0
yield_curve        0
oil                0
gold               0
credit_spread      0
monetary_policy    0
volatility         0
stock_bond_corr    0
dtype: int64

First rows:
             sp500 

In [10]:
print(data.shape)
print(data.index[0], data.index[-1])
print(data.describe())

(468, 8)
1986-01-31 00:00:00 2024-12-31 00:00:00
             sp500  yield_curve         oil        gold  credit_spread  \
count   468.000000   468.000000  468.000000  468.000000     468.000000   
mean   1553.713632     1.590577   47.797500   32.541667       2.291517   
std    1267.260026     1.207926   29.723022   22.537700       0.694981   
min     211.780000    -1.570000   10.250000    9.900000       1.300000   
25%     583.682500     0.667500   20.285000   14.100000       1.770000   
50%    1219.610000     1.600000   40.985000   18.100000       2.175000   
75%    2024.522500     2.570000   71.515000   49.025000       2.672500   
max    6032.380000     3.760000  139.960000  102.400000       6.100000   

       monetary_policy  volatility  stock_bond_corr  
count       468.000000  468.000000       468.000000  
mean          3.079808   19.250562         0.028748  
std           2.453290    8.189105         0.070103  
min           0.010000    8.359639        -0.125293  
25%           

In [14]:
import urllib
import zipfile
import io
def download_ff5() -> pd.DataFrame:
    url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip"
    print("Downloading FF5 factors...")
    with urllib.request.urlopen(url) as r:
        zf = zipfile.ZipFile(io.BytesIO(r.read()))
        # print filenames to see what's actually in the zip
        print("Files in zip:", zf.namelist())
        fname = zf.namelist()[0]   # just take the first file
        with zf.open(fname) as f:
            raw = f.read().decode("utf-8").split("\n")

    start = next(i for i, line in enumerate(raw) if line.strip()[:6].isdigit())
    end   = next(i for i, line in enumerate(raw[start:], start)
                 if line.strip() == "" or (line.strip() and not line.strip()[:6].isdigit()))

    df = pd.read_csv(
        io.StringIO("\n".join(raw[start:end])),
        header=None,
        names=["date", "Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"],
    )
    df["date"] = pd.to_datetime(df["date"].astype(str).str.strip(), format="%Y%m")
    df         = df.set_index("date")
    df         = df.apply(pd.to_numeric, errors="coerce") / 100
    df.index   = df.index + pd.offsets.MonthEnd(0)
    return df.dropna()

ff5 = download_ff5()
print(ff5.head())
print(ff5.shape)

Files in zip: ['F-F_Research_Data_5_Factors_2x3.csv']
            Mkt-RF     SMB     HML     RMW     CMA      RF
date                                                      
1963-07-31 -0.0039 -0.0048 -0.0081  0.0064 -0.0115  0.0027
1963-08-31  0.0508 -0.0080  0.0170  0.0040 -0.0038  0.0025
1963-09-30 -0.0157 -0.0043  0.0000 -0.0078  0.0015  0.0027
1963-10-31  0.0254 -0.0134 -0.0004  0.0279 -0.0225  0.0029
1963-11-30 -0.0086 -0.0085  0.0173 -0.0043  0.0227  0.0027
(752, 6)


In [15]:
def download_momentum() -> pd.Series:
    url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_CSV.zip"
    print("Downloading Momentum factor...")
    with urllib.request.urlopen(url) as r:
        zf = zipfile.ZipFile(io.BytesIO(r.read()))
        print("Files in zip:", zf.namelist())
        fname = zf.namelist()[0]
        with zf.open(fname) as f:
            raw = f.read().decode("utf-8").split("\n")

    start = next(i for i, line in enumerate(raw) if line.strip()[:6].isdigit())
    end   = next(i for i, line in enumerate(raw[start:], start)
                 if line.strip() == "" or (line.strip() and not line.strip()[:6].isdigit()))

    df = pd.read_csv(
        io.StringIO("\n".join(raw[start:end])),
        header=None,
        names=["date", "Mom"],
    )
    df["date"] = pd.to_datetime(df["date"].astype(str).str.strip(), format="%Y%m")
    df         = df.set_index("date")
    df["Mom"]  = pd.to_numeric(df["Mom"], errors="coerce") / 100
    df.index   = df.index + pd.offsets.MonthEnd(0)
    return df["Mom"].dropna()

mom = download_momentum()
print(mom.head())
print(mom.shape)

Files in zip: ['F-F_Momentum_Factor.csv']
date
1927-01-31    0.0057
1927-02-28   -0.0150
1927-03-31    0.0352
1927-04-30    0.0436
1927-05-31    0.0278
Name: Mom, dtype: float64
(1190,)


In [16]:
factors = ff5[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]].join(mom, how="inner")
factors.columns = ["Market", "Size", "Value", "Profitability", "Investment", "Momentum"]
print(factors.shape)
print(factors.index[0], factors.index[-1])

(752, 6)
1963-07-31 00:00:00 2026-02-28 00:00:00


In [17]:
def download_ff_industries() -> pd.DataFrame:
    """Download FF 12 Industry Portfolios from Ken French."""
    url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/12_Industry_Portfolios_CSV.zip"
    print("Downloading FF 12 Industry Portfolios...")
    with urllib.request.urlopen(url) as r:
        zf = zipfile.ZipFile(io.BytesIO(r.read()))
        print("Files in zip:", zf.namelist())
        fname = zf.namelist()[0]
        with zf.open(fname) as f:
            raw = f.read().decode("utf-8").split("\n")

    # find data start
    start = next(i for i, line in enumerate(raw)
                 if line.strip()[:6].isdigit())
    # find end (blank line or annual section)
    end = next(i for i, line in enumerate(raw[start:], start)
               if line.strip() == "" or
               (line.strip() and not line.strip()[:6].isdigit()))

    df = pd.read_csv(io.StringIO("\n".join(raw[start:end])), header=None)
    df.columns = ["date"] + list(df.columns[1:])
    df["date"] = pd.to_datetime(df["date"].astype(str).str.strip(), format="%Y%m")
    df = df.set_index("date")
    df = df.apply(pd.to_numeric, errors="coerce") / 100
    df.index = df.index + pd.offsets.MonthEnd(0)

    # name the 12 industries
    df.columns = ["NoDur", "Durbl", "Manuf", "Enrgy",
                  "HiTec", "Telcm", "Shops", "Hlth",
                  "Utils", "Other", "Finance", "Other2"]
    return df.dropna()

industries = download_ff_industries()
print(f"\nIndustries shape: {industries.shape}")
print(f"Date range: {industries.index[0].date()} → {industries.index[-1].date()}")
print(f"\nColumns: {list(industries.columns)}")
print(f"\nSample:\n{industries.head()}")

Files in zip: ['12_Industry_Portfolios.csv']

Industries shape: (1196, 12)
Date range: 1926-07-31 → 2026-02-28

Columns: ['NoDur', 'Durbl', 'Manuf', 'Enrgy', 'HiTec', 'Telcm', 'Shops', 'Hlth', 'Utils', 'Other', 'Finance', 'Other2']

Sample:
             NoDur   Durbl   Manuf   Enrgy   HiTec   Telcm   Shops    Hlth  \
date                                                                         
1926-07-31  0.0144  0.1390  0.0367 -0.0114  0.0802  0.0315  0.0083  0.0704   
1926-08-31  0.0399  0.0370  0.0241  0.0343  0.0510  0.0198  0.0217 -0.0170   
1926-09-30  0.0115  0.0498 -0.0007 -0.0330  0.0538 -0.0035  0.0242  0.0205   
1926-10-31 -0.0124 -0.0839 -0.0318 -0.0078 -0.0455 -0.0538 -0.0011 -0.0327   
1926-11-30  0.0521 -0.0017  0.0403  0.0001  0.0509  0.0479  0.0163  0.0440   

             Utils   Other  Finance  Other2  
date                                         
1926-07-31  0.0012  0.0185  -0.0002  0.0223  
1926-08-31 -0.0072  0.0417   0.0426  0.0435  
1926-09-30  0.0021  0.0069  

In [18]:
# At the end of the data cell, add:
factors.to_csv("data/ff_factors_monthly.csv")
industries.to_csv("data/ff_industries_monthly.csv")
print("Saved ff_factors_monthly.csv and ff_industries_monthly.csv")

Saved ff_factors_monthly.csv and ff_industries_monthly.csv
